In [ ]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, Runtime
from langchain.messages import AIMessage

from pretty_print_agent_result import pprint


class PolicyGuardMiddleware(AgentMiddleware):
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"[before_agent] 对话开始， 消息数: {len(state['messages'])}")
        return None

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"[before_model] 模型调用开始， 消息数: {len(state['messages'])}")
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        last = state["messages"][-1]
        if isinstance(last, AIMessage) and "BLOCKED_CN" in (last.content or ""):
            print("[after_model] 触发安全规则: 检测到 'BLOCKWED_CN', 跳转 end")
        return {"messages": [AIMessage("改请求触发了安全校验，无法继续。")], "jump_to": "end"}

    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"[after_agent] 对话结束 消息数: {len(state['messages'])}")


import time
from typing import Callable

from langchain.agents.middleware import ModelRequest, ModelResponse


class RetryMetricsMiddleware(AgentMiddleware):
    def __init__(self, max_retries: int = 2, base_delay: float = 0.1):
        self.max_retries = max_retries
        self.base_delay = base_delay

    def wrap_model_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
        start = time.time()
        try:
            for i in range(self.max_retries + 1):
                try:
                    return handler(request)
                except Exception:
                    if i == self.max_retries:
                        raise
                    backoff = self.base_delay * 2**i
                    print(f"[wrap_model_call] 调用失败， 将在 {backoff}s 后重试")
                    time.sleep(backoff)

        finally:
            cost = time.time() - start
            print(f"[wrap_model_call] 调用结束， 耗时: {cost:.3f}s")


import os

import dotenv
from langchain.chat_models import init_chat_model

dotenv.load_dotenv()
model = init_chat_model(
    model="deepseek-chat",
    base_url="https://api.deepseek.com/v1",
    api_key=os.getenv("DEEPSEEK_API"),
    temperature=0.7,
    model_provider="openai",
)

from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        RetryMetricsMiddleware(max_retries=3, base_delay=0.2),
        PolicyGuardMiddleware(),
    ],
)

from langchain.messages import HumanMessage

res1 = agent.invoke({"messages": [HumanMessage(content="用一句话解释 LangChain")]})
await pprint(result=res1, question="用一句话解释 LangChain")

res2 = agent.invoke({"messages": HumanMessage(content="请只回复: BLOCKED_CN")})
await pprint(result=res2, question="用一句话解释 LangChain")

[before_agent] 对话开始， 消息数: 1
[before_model] 模型调用开始， 消息数: 1
[wrap_model_call] 调用结束， 耗时: 1.444s
[after_agent] 对话结束 消息数: 3


╭──────────────────────────────────────────── 用一句话解释 LangChain ─────────────────────────────────────────────╮
│ ╭────────────────────────────────────────────────── Answer ───────────────────────────────────────────────────╮ │
│ │ 改请求触发了安全校验，无法继续。                                                                            │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│ ╭───────────────────────────────────────────────── Messages ──────────────────────────────────────────────────╮ │
│ │ ╭──────────────────────────────────────────── 0: HumanMessage ────────────────────────────────────────────╮ │ │
│ │ │ {                                                                                                       │ │ │
│ │ │   "content": "用一句话解释 LangChain",                                                                  │ │ │
│ │ │   "additional_kwargs": {},                                                                              │ │ │
│ │ │   "response_metadata": {},                                                                              │ │ │
│ │ │   "type": "human",                                                                                      │ │ │
│ │ │   "name": null,                                                                                         │ │ │
│ │ │   "id": "780d15c3-1f56-46fc-ae8d-f3644e4dc69d"                                                          │ │ │
│ │ │ }                                                                                                       │ │ │
│ │ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │ │
│ │ ╭───────────────────────────────────────────── 1: AIMessage ──────────────────────────────────────────────╮ │ │
│ │ │ {                                                                                                       │ │ │
│ │ │   "content":                                                                                            │ │ │
│ │ │ "LangChain是一个开源框架，通过提供统一的接口和工具链，帮助开发者将大型语言模型（如GPT）与外部数据源、AP │ │ │
│ │ │ I及自定义逻辑无缝集成，从而构建复杂的AI应用。",                                                         │ │ │
│ │ │   "additional_kwargs": {                                                                                │ │ │
│ │ │     "refusal": null                                                                                     │ │ │
│ │ │   },                                                                                                    │ │ │
│ │ │   "response_metadata": {                                                                                │ │ │
│ │ │     "token_usage": {                                                                                    │ │ │
│ │ │       "completion_tokens": 42,                                                                          │ │ │
│ │ │       "prompt_tokens": 9,                                                                               │ │ │
│ │ │       "total_tokens": 51,                                                                               │ │ │
│ │ │       "completion_tokens_details": null,                                                                │ │ │
│ │ │       "prompt_tokens_details": {                                                                        │ │ │
│ │ │         "audio_tokens": null,                                                                           │ │ │
│ │ │         "cached_tokens": 0                                                                              │ │ │
│ │ │       },                                                                                                │ │ │
│ │ │       "prompt_cache_hit_tokens": 0,                                                                     │ │ │
│ │ │       "prompt_cache_miss_tokens": 9                                                                     │ │ │
│ │ │     },                     

[before_agent] 对话开始， 消息数: 1
[before_model] 模型调用开始， 消息数: 1
[wrap_model_call] 调用结束， 耗时: 1.046s
[after_model] 触发安全规则: 检测到 'BLOCKWED_CN', 跳转 end
[after_agent] 对话结束 消息数: 3


╭──────────────────────────────────────────── 用一句话解释 LangChain ─────────────────────────────────────────────╮
│ ╭────────────────────────────────────────────────── Answer ───────────────────────────────────────────────────╮ │
│ │ 改请求触发了安全校验，无法继续。                                                                            │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│ ╭───────────────────────────────────────────────── Messages ──────────────────────────────────────────────────╮ │
│ │ ╭──────────────────────────────────────────── 0: HumanMessage ────────────────────────────────────────────╮ │ │
│ │ │ {                                                                                                       │ │ │
│ │ │   "content": "请只回复: BLOCKED_CN",                                                                    │ │ │
│ │ │   "additional_kwargs": {},                                                                              │ │ │
│ │ │   "response_metadata": {},                                                                              │ │ │
│ │ │   "type": "human",                                                                                      │ │ │
│ │ │   "name": null,                                                                                         │ │ │
│ │ │   "id": "aaaaa8fc-64f1-48d0-ad7e-ca2cf45efdeb"                                                          │ │ │
│ │ │ }                                                                                                       │ │ │
│ │ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │ │
│ │ ╭───────────────────────────────────────────── 1: AIMessage ──────────────────────────────────────────────╮ │ │
│ │ │ {                                                                                                       │ │ │
│ │ │   "content": "BLOCKED_CN",                                                                              │ │ │
│ │ │   "additional_kwargs": {                                                                                │ │ │
│ │ │     "refusal": null                                                                                     │ │ │
│ │ │   },                                                                                                    │ │ │
│ │ │   "response_metadata": {                                                                                │ │ │
│ │ │     "token_usage": {                                                                                    │ │ │
│ │ │       "completion_tokens": 5,                                                                           │ │ │
│ │ │       "prompt_tokens": 13,                                                                              │ │ │
│ │ │       "total_tokens": 18,                                                                               │ │ │
│ │ │       "completion_tokens_details": null,                                                                │ │ │
│ │ │       "prompt_tokens_details": {                                                                        │ │ │
│ │ │         "audio_tokens": null,                                                                           │ │ │
│ │ │         "cached_tokens": 0                                                                              │ │ │
│ │ │       },                                                                                                │ │ │
│ │ │       "prompt_cache_hit_tokens": 0,                                                                     │ │ │
│ │ │       "prompt_cache_miss_tokens": 13                                                                    │ │ │
│ │ │     },                                                                                                  │ │ │
│ │ │     "model_provider": "openai",                                             